## 01. Create Discovery Golden Dataset

### Why are we implementing this?

We need a separate golden dataset to evaluate whether SEVA_AI can identify relevant government schemes from a citizen's real-world problem.

Unlike our existing FAQ dataset, citizen queries must not contain scheme names.

Each query will have manually verified relevance judgments for multiple schemes.

We will use this dataset to evaluate:
- Scheme discovery accuracy
- Recall@K and Precision@K
- MRR and MAP
- nDCG with graded relevance

We will initially target 150 citizen queries.

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

DISCOVERY_DATASET_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "golden_dataset"
    / "discovery"
)

DISCOVERY_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_golden_dataset.jsonl"
)

SCHEMA_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_schema.json"
)

schema = {
    "query_id": "Unique query identifier",
    "citizen_query": "Citizen's actual problem without scheme names",
    "category": "Government scheme category",
    "answerable": "Boolean",
    "relevant_scheme_ids": [
        "Verified relevant scheme IDs"
    ],
    "relevance_judgments": [
        {
            "scheme_id": "Government scheme ID",
            "relevance_grade": 2,
            "rationale": "Why this scheme is relevant",
            "source_url": "Official verification URL"
        }
    ]
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

if not DATASET_PATH.exists():
    DATASET_PATH.touch()

print("Discovery benchmark initialized!")
print("Dataset:", DATASET_PATH)
print("Schema:", SCHEMA_PATH)
print("Target queries: 150")

Discovery benchmark initialized!
Dataset: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_golden_dataset.jsonl
Schema: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_schema.json
Target queries: 150


## 02. Analyze Scheme Categories

### Why are we implementing this?

We need to understand the distribution of government schemes before creating our 150 citizen queries.

We will inspect the existing scheme dataset to identify:
- Available scheme categories
- Number of schemes per category
- Available metadata fields
- Sample scheme records

This analysis will help us create a balanced discovery benchmark across different government domains.

In [3]:
import json
from collections import Counter
from pathlib import Path

# SCHEME_PATH = (
#     PROJECT_ROOT
#     / "notebooks"
#     / "golden_dataset"
#     / "selected_schemes.json"
# )

SCHEME_PATH ="G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json"

with open(SCHEME_PATH, "r", encoding="utf-8") as f:
    scheme_data = json.load(f)

if isinstance(scheme_data, list):
    schemes = scheme_data
elif isinstance(scheme_data, dict):
    schemes = scheme_data.get("schemes", [])
else:
    raise ValueError("Unexpected scheme dataset structure")

print("Total schemes:", len(schemes))

if not schemes:
    raise ValueError("No scheme records found.")

print("\nAvailable fields:")
print(list(schemes[0].keys()))

category_counts = Counter()

for scheme in schemes:
    categories = scheme.get("categories", [])

    if isinstance(categories, str):
        categories = [categories]

    category_counts.update(
        category for category in categories if category
    )

print("\nCategory distribution:")

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nSample scheme:")
print(json.dumps(schemes[0], indent=2, ensure_ascii=False))

Total schemes: 4428

Available fields:
['scheme_id', 'internal_id', 'slug', 'name', 'short_title', 'level', 'ministry', 'nodal_department', 'department', 'states', 'categories', 'tags', 'target_beneficiaries', 'status', 'open_date', 'close_date', 'brief_description', 'details', 'benefits', 'exclusions', 'eligibility', 'application_process', 'documents_required', 'faqs', 'apply_url', 'apply_links', 'official_links', 'myscheme_url', 'source', 'fetched_on']

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Sample scheme:
{
  "scheme_id": "LBezF5gB5uF80_MGGY81",
  "internal_id": "628727

## Full Scheme Dataset Validation

Validate the complete government scheme dataset before building the citizen-discovery benchmark. This checks data integrity, category coverage, scheme status, and geographic distribution.

In [4]:
import json
from pathlib import Path
from collections import Counter

SCHEME_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\data\analysis\all_schemes.json")

with open(SCHEME_PATH, "r", encoding="utf-8") as file:
    schemes = json.load(file)

print("Total schemes:", len(schemes))
print("Unique scheme IDs:", len({s.get("scheme_id") for s in schemes}))
print("Categories:", len({
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
}))

category_counts = Counter(
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
)

status_counts = Counter(
    scheme.get("status", "unknown")
    for scheme in schemes
)

level_counts = Counter(
    scheme.get("level", "unknown")
    for scheme in schemes
)

state_counts = Counter(
    state
    for scheme in schemes
    for state in scheme.get("states", [])
)

print("\nCategory distribution:")
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nStatus distribution:", status_counts)
print("\nLevel distribution:", level_counts)
print("\nTop 15 states:", state_counts.most_common(15))

Total schemes: 4428
Unique scheme IDs: 4428
Categories: 15

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Status distribution: Counter({'active': 4384, 'closed': 44})

Level distribution: Counter({'State': 3735, 'Central': 693})

Top 15 states: [('All', 673), ('Gujarat', 594), ('Uttarakhand', 416), ('Madhya Pradesh', 263), ('Goa', 247), ('Puducherry', 245), ('Haryana', 224), ('Tamil Nadu', 212), ('Rajasthan', 147), ('West Bengal', 105), ('Chhattisgarh', 103), ('Bihar', 102), ('Assam', 86), ('Jharkhand', 86), ('Meghalaya', 85)]


## Geographic Coverage Analysis

Analyze scheme availability by state and scheme level. This helps us design citizen queries that test both Central and State government scheme discovery, including location-specific eligibility.

In [5]:
from collections import Counter

central_schemes = [
    s for s in schemes if s.get("level") == "Central"
]

state_schemes = [
    s for s in schemes if s.get("level") == "State"
]

state_distribution = Counter(
    state
    for scheme in state_schemes
    for state in scheme.get("states", [])
    if state != "All"
)

print("Central schemes:", len(central_schemes))
print("State schemes:", len(state_schemes))

print("\nState-wise scheme distribution:")
for state, count in state_distribution.most_common():
    print(f"{state}: {count}")

print("\nCentral scheme state values:")
print(Counter(
    state
    for scheme in central_schemes
    for state in scheme.get("states", [])
))

Central schemes: 693
State schemes: 3735

State-wise scheme distribution:
Gujarat: 592
Uttarakhand: 415
Madhya Pradesh: 261
Goa: 246
Puducherry: 245
Haryana: 224
Tamil Nadu: 206
Rajasthan: 145
West Bengal: 104
Chhattisgarh: 101
Bihar: 101
Jharkhand: 85
Meghalaya: 79
Kerala: 78
Assam: 77
Maharashtra: 77
Odisha: 77
Himachal Pradesh: 73
Dadra & Nagar Haveli and Daman & Diu: 59
Karnataka: 54
Delhi: 49
Andhra Pradesh: 44
Punjab: 41
Uttar Pradesh: 41
Jammu and Kashmir: 39
Tripura: 36
Arunachal Pradesh: 35
Manipur: 24
Sikkim: 21
Mizoram: 21
Telangana: 21
Chandigarh: 19
Andaman and Nicobar Islands: 17
Nagaland: 16
Lakshadweep: 9
Ladakh: 3

Central scheme state values:
Counter({'All': 673, 'Assam': 9, 'Tripura': 7, 'Arunachal Pradesh': 7, 'Manipur': 7, 'Mizoram': 7, 'Tamil Nadu': 6, 'Sikkim': 6, 'Meghalaya': 6, 'Kerala': 6, 'Nagaland': 6, 'Karnataka': 4, 'Jammu and Kashmir': 3, 'Gujarat': 2, 'Rajasthan': 2, 'Madhya Pradesh': 2, 'Chhattisgarh': 2, 'Odisha': 2, 'Maharashtra': 1, 'West Bengal': 1,

## National Geographic Coverage Inventory

Prepare a complete state and union territory coverage inventory from the 4,428-scheme dataset. This will help distribute the citizen-discovery benchmark across India without prioritizing one state.

In [6]:
from collections import Counter

state_scheme_counts = Counter(
    state
    for scheme in schemes
    if scheme.get("level") == "State"
    for state in scheme.get("states", [])
    if state != "All"
)

print("Total states and UTs:", len(state_scheme_counts))
print("\nNational geographic coverage:\n")

for index, (state, count) in enumerate(
    sorted(state_scheme_counts.items()), start=1
):
    print(f"{index:02d}. {state}: {count} schemes")

Total states and UTs: 36

National geographic coverage:

01. Andaman and Nicobar Islands: 17 schemes
02. Andhra Pradesh: 44 schemes
03. Arunachal Pradesh: 35 schemes
04. Assam: 77 schemes
05. Bihar: 101 schemes
06. Chandigarh: 19 schemes
07. Chhattisgarh: 101 schemes
08. Dadra & Nagar Haveli and Daman & Diu: 59 schemes
09. Delhi: 49 schemes
10. Goa: 246 schemes
11. Gujarat: 592 schemes
12. Haryana: 224 schemes
13. Himachal Pradesh: 73 schemes
14. Jammu and Kashmir: 39 schemes
15. Jharkhand: 85 schemes
16. Karnataka: 54 schemes
17. Kerala: 78 schemes
18. Ladakh: 3 schemes
19. Lakshadweep: 9 schemes
20. Madhya Pradesh: 261 schemes
21. Maharashtra: 77 schemes
22. Manipur: 24 schemes
23. Meghalaya: 79 schemes
24. Mizoram: 21 schemes
25. Nagaland: 16 schemes
26. Odisha: 77 schemes
27. Puducherry: 245 schemes
28. Punjab: 41 schemes
29. Rajasthan: 145 schemes
30. Sikkim: 21 schemes
31. Tamil Nadu: 206 schemes
32. Telangana: 21 schemes
33. Tripura: 36 schemes
34. Uttar Pradesh: 41 schemes
35. 

## National Discovery Benchmark Allocation

Create a balanced allocation of 150 citizen queries across all 15 scheme categories and 36 Indian States and Union Territories. Include both Central and State scheme scenarios while ensuring every state and UT is represented.

In [7]:
import pandas as pd
from collections import Counter

state_counts = {
    state: count
    for state, count in state_scheme_counts.items()
}

# Every state/UT gets at least one query.
# Distribute the remaining 64 queries proportionally
# to the square root of scheme coverage to reduce dominance.
import numpy as np

states = list(state_counts)
weights = np.sqrt([state_counts[state] for state in states])

remaining = 100 - len(states)
quotas = weights / weights.sum() * remaining

allocations = np.floor(quotas).astype(int)
leftover = remaining - allocations.sum()

remainders = quotas - allocations
for index in np.argsort(-remainders)[:leftover]:
    allocations[index] += 1

state_allocation = {
    state: 1 + int(allocation)
    for state, allocation in zip(states, allocations)
}

allocation_df = pd.DataFrame([
    {
        "state_ut": state,
        "available_schemes": state_counts[state],
        "planned_queries": state_allocation[state]
    }
    for state in states
]).sort_values("state_ut")

print("Total state queries:", allocation_df["planned_queries"].sum())
print("States/UTs covered:", len(allocation_df))
display(allocation_df)

print("\nTotal Central scenarios: 50")
print("Total State scenarios:", allocation_df["planned_queries"].sum())
print("Total benchmark queries:",
      50 + allocation_df["planned_queries"].sum())

Total state queries: 100
States/UTs covered: 36


,state_ut,available_schemes,planned_queries
29,Andaman and Nicobar Islands,17,2
31,Andhra Pradesh,44,2
15,Arunachal Pradesh,35,2
7,Assam,77,3
19,Bihar,101,3
26,Chandigarh,19,2
16,Chhattisgarh,101,3
30,Dadra & Nagar Haveli and Daman & Diu,59,3
0,Delhi,49,2
23,Goa,246,4



Total Central scenarios: 50
Total State scenarios: 100
Total benchmark queries: 150


In [8]:
import pandas as pd

records = []

for scheme in schemes:
    if scheme.get("level") != "State":
        continue

    for state in scheme.get("states", []):
        if state == "All":
            continue

        for category in scheme.get("categories", []):
            records.append({
                "state": state,
                "category": category,
                "scheme_id": scheme.get("scheme_id")
            })

state_category_df = pd.DataFrame(records)

coverage_matrix = pd.crosstab(
    state_category_df["state"],
    state_category_df["category"]
)

print("State-category combinations:", len(state_category_df))
display(coverage_matrix)

print("\nCategories with no State schemes:")
print(
    coverage_matrix.columns[
        coverage_matrix.sum(axis=0) == 0
    ].tolist()
)

State-category combinations: 5003


category,"Agriculture,Rural & Environment","Banking,Financial Services and Insurance",Business & Entrepreneurship,Education & Learning,Health & Wellness,Housing & Shelter,"Public Safety,Law & Justice","Science, IT & Communications",Skills & Employment,Social welfare & Empowerment,Sports & Culture,Transport & Infrastructure,Travel & Tourism,Utility & Sanitation,Women and Child
state,,,,,,,,,,,,,,,
Andaman and Nicobar Islands,0,1,0,3,2,0,0,0,0,14,0,0,1,1,3
Andhra Pradesh,0,0,4,10,2,1,0,0,3,32,0,0,0,0,8
Arunachal Pradesh,12,4,3,4,5,1,0,0,2,10,0,1,2,1,5
Assam,11,9,3,23,11,0,1,0,6,29,4,2,1,0,3
Bihar,22,4,8,21,9,3,0,0,7,42,0,2,0,1,14
Chandigarh,0,1,0,1,2,0,0,0,0,17,1,0,0,0,7
Chhattisgarh,12,0,7,29,13,0,0,0,5,36,1,2,2,1,11
Dadra & Nagar Haveli and Daman & Diu,5,0,28,17,2,3,0,0,0,12,0,0,0,0,9
Delhi,0,3,7,16,4,2,0,0,1,23,0,2,1,0,6



Categories with no State schemes:
[]


One important distinction: 10% is our sampling target, not necessarily one unique question per scheme. We should generate questions from scheme information, then verify that each question has the correct relevant scheme IDs. Some citizen questions may legitimately match multiple schemes.

In [9]:
import math
import pandas as pd

state_targets = pd.DataFrame([
    {
        "state": state,
        "scheme_count": count,
        "target_questions": math.ceil(count * 0.10)
    }
    for state, count in state_scheme_counts.items()
])

central_count = len(central_schemes)
central_target = math.ceil(central_count * 0.10)

print(state_targets.to_string(index=False))
print("\nState questions:", state_targets["target_questions"].sum())
print("Central questions:", central_target)
print(
    "Total target questions:",
    state_targets["target_questions"].sum() + central_target
)

                               state  scheme_count  target_questions
                               Delhi            49                 5
                              Sikkim            21                 3
                             Haryana           224                23
                           Karnataka            54                 6
                             Mizoram            21                 3
                    Himachal Pradesh            73                 8
                             Gujarat           592                60
                               Assam            77                 8
                           Jharkhand            85                 9
                            Nagaland            16                 2
                             Manipur            24                 3
                           Rajasthan           145                15
                         West Bengal           104                11
                         Maharasht

## Discovery Benchmark Scheme Sampling

Select approximately 10% of State schemes for each State/UT and 10% of Central schemes. Each selected scheme becomes a source for generating one natural-language citizen query. Preserve the original scheme information and geographic scope for relevance judgments.

In [10]:
import json
import math
import random
from pathlib import Path

random.seed(42)

discovery_dir = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)
discovery_dir.mkdir(parents=True, exist_ok=True)

generation_tasks = []

# Sample State schemes separately for each State/UT
for state, count in state_scheme_counts.items():
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Sample Central schemes
central_target = math.ceil(len(central_schemes) * 0.10)
selected_central = random.sample(
    central_schemes, central_target
)

for scheme in selected_central:
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

output_path = discovery_dir / "discovery_generation_tasks.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    for task in generation_tasks:
        file.write(
            json.dumps(task, ensure_ascii=False) + "\n"
        )

print("Generation tasks:", len(generation_tasks))
print("State tasks:", sum(t["scope"] == "State" for t in generation_tasks))
print("Central tasks:", sum(t["scope"] == "Central" for t in generation_tasks))
print("Saved:", output_path)

Generation tasks: 462
State tasks: 392
Central tasks: 70
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_generation_tasks.jsonl
